# 0. Setup

In [1]:
import ibis
from ibis import _
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str, reindex_entity, add_fe

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

# First difference the panel data

In [2]:
%%script fd sample
g_name = 'working_yearly_g'
n_name = 'working_yearly_n'
t_panel_g = con.table(g_name)
t_panel_n = con.table(n_name)

#--- #
t_panel = add_fe(t_panel_g, fe=['t', 'i', 'c'])
t_panel_sample = (
    t_panel
    .select([c for c in t_panel.columns if c.startswith('year_')])
    .order_by(ibis.random())
    .limit(5)
)
display(t_panel_sample.execute())

Couldn't find program: 'fd'


# 1a. LMM
- 26 seconds run nowadays estimating 2 models

In [3]:
panel_name = "working_yearly_g"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models_1 = {
    'lmm_exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i', 'c'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l'
            ],
        },
        'description': 'Instrument endogenous effect, LMM'
    }
}
models_3 = {
    'lmm_3_rings': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'W': ['wg2_y', 'wg2_k', 'wg2_l', 'wg3_l', 'wg3_k', 'wg3_y'],
        'fe': ['t', 'i', 'c'],
        'description': 'Strict exogeneity, structural form',
        'include': True
    },
    'lmm_3_rings_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l']
        },
        'description': '3-level regressors, squared instruments',
        'include': True
    },
    'lmm_3_rings_more_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 'c', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l'],#, 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l'],#, 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l']#, 'wg3wg2_k', 'wg3wg2_l']
        },
        'description': '3-level regressors, many instruments',
        'include': False
    }
}

models = models_1
out_name = "results_1a_lmm_pc8"
# out_name = "results_1b_lmm_3r"

# 5m run for 3 complicated IV models
run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

Running model 'lmm_exog' as panel OLS
✅ Model 'lmm_exog' estimated: k=0.261, l=0.650, wg1_y=0.199, wg1_k=-0.057, wg1_l=-0.118
Running model 'lmm_instr' as linearmodels panel IV, formula: y ~ k + l + wg1_k + wg1_l + year_2011 + year_2008 + year_2015 + year_2014 + year_2013 + year_2021 + year_2023 + year_2024 + year_2017 + year_2022 + year_2007 + year_2010 + year_2009 + year_2016 + year_2018 + year_2019 + year_2020 + year_2012 + [wg1_y ~ w2g1_k + w2g1_l + w3g1_k + w3g1_l]
J-statistic (rej if overidentified): 3.43, p-value: 0.330
✅ Model 'lmm_instr' estimated: k=0.233, l=0.555, wg1_y=-0.384, wg1_k=0.118, wg1_l=0.249
Panel regressions complete. 2 models, writing to results_1a_lmm_pc8.txt


# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [5]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

panel_name = "working_yearly_n"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
m_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 1 model'
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 2 model'
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 3 model'
    }
}
deeper_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': '2nd-order spatial lag model'
    },
    'dd1-3lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'description': '3rd-order spatial lag model'
    }
}

models = deeper_models
out_name = "results_2b_dd_deeper"

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

Running model 'dd1' as linearmodels panel IV, formula: y ~ k + l + wd1_k + wd1_l + year_2016 + year_2008 + year_2015 + year_2014 + year_2012 + year_2010 + year_2009 + year_2013 + year_2007 + year_2022 + year_2011 + year_2019 + year_2018 + year_2020 + year_2017 + year_2021 + year_2023 + year_2024 + [wd1_y ~ w2d1_k + w2d1_l]
J-statistic (rej if overidentified): 1.03, p-value: 0.311
✅ Model 'dd1' estimated: k=0.271, l=0.512, wd1_y=-0.159, wd1_k=0.054, wd1_l=0.105
Running model 'dd1-3lag' as linearmodels panel IV, formula: y ~ k + l + wd1_k + wd1_l + year_2013 + year_2020 + year_2012 + year_2017 + year_2016 + year_2009 + year_2010 + year_2007 + year_2022 + year_2008 + year_2018 + year_2019 + year_2011 + year_2015 + year_2014 + year_2021 + year_2023 + year_2024 + [wd1_y ~ w2d1_k + w2d1_l + w3d1_k + w3d1_l]
J-statistic (rej if overidentified): 6.26, p-value: 0.100
✅ Model 'dd1-3lag' estimated: k=0.267, l=0.513, wd1_y=-0.740, wd1_k=0.250, wd1_l=0.478
Panel regressions complete. 2 models, writ